# Notebook 1 — Patch Lens Attribution: k×k Neighborhood Scoring

**What this notebook does:**
For each patch position `(i, j)` in the 16×16 grid at each target ViT layer `L`:

1. Extract the **k×k neighborhood** around `(i, j)` from the patch token grid  
2. Flatten to `[k*k*d_model]`  
3. Feed to the **patch lens** trained on that neighborhood size  
4. `score[i,j] = softmax(logits)[y_hat]` ∈ [0, 1]

Border patches (within `patch_border` rows/cols of the edge) are **NaN** because the lens was not trained with full neighborhoods for those positions.

Sanity-check cells (**SC N**) follow every major step to verify shapes, value ranges, and logic.

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.datasets as datasets

# notebooks/ lives next to src/
sys.path.insert(0, str(Path('..').resolve() / 'src'))

from tuned_lens.model import VisionModelWrapper
from tuned_lens.config import ModelConfig
from attribute_lens.scorer import load_lens_checkpoint, discover_lens_files

print('Imports OK')

In [ ]:
# ── CONFIGURATION — edit paths to match your environment ─────────────────────
IMAGENET_VAL_DIR  = '/opt/models/datasets/imagenet/extracted/val'
PATCH_LENS_DIR    = '../outputs/patch_affine_kld/best_lenses'
MODEL_NAME        = 'vit_large_patch14_clip_224.openai_ft_in1k'

TARGET_LAYERS       = [0, 6, 12, 18, 23]  # will be filtered to available checkpoints
PATCH_NEIGHBOR_SIZE = 3                    # k for k×k neighborhood (must match training)
PATCH_BORDER        = 2                    # border patches have NaN score
N_IMAGES            = 10
SEED                = 42
DEVICE              = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Device   : {DEVICE}')
print(f'Lens dir : {PATCH_LENS_DIR}')

## SC 1 — Verify paths and discover available checkpoint layers

In [ ]:
assert Path(IMAGENET_VAL_DIR).exists(), f'Val dir not found: {IMAGENET_VAL_DIR}'
assert Path(PATCH_LENS_DIR).exists(),   f'Lens dir not found: {PATCH_LENS_DIR}'

available_lenses = discover_lens_files(PATCH_LENS_DIR)
print(f'Found {len(available_lenses)} patch-lens checkpoints')
print(f'  Available layers : {sorted(available_lenses.keys())}')

# Keep only requested layers that have a checkpoint file
TARGET_LAYERS = sorted(l for l in TARGET_LAYERS if l in available_lenses)
assert TARGET_LAYERS, (
    f'None of the requested layers have checkpoints. '
    f'Found: {sorted(available_lenses.keys())}'
)
print(f'  Using layers     : {TARGET_LAYERS}')
print('SC 1 ✓')

## Load model (patch_mode=True extracts all patch tokens, not just CLS)

In [ ]:
model_cfg = ModelConfig(
    model_name=MODEL_NAME,
    pretrained=True,
    target_layers=TARGET_LAYERS,
    freeze_model=True,
    patch_mode=True,   # hooks capture patch tokens (positions 1:) from each block
)
wrapper = VisionModelWrapper(model_cfg, device=DEVICE)

H, W = wrapper.patch_grid_size
print(f'Model      : {MODEL_NAME}')
print(f'd_model    : {wrapper.d_model}')
print(f'num_classes: {wrapper.num_classes}')
print(f'num_layers : {wrapper.num_layers}')
print(f'patch_grid : {H}x{W} = {H * W} patches per image')
print(f'target_lyrs: {wrapper.target_layers}')

## SC 2 — Verify model properties

In [ ]:
assert wrapper.d_model > 0,                   'd_model must be positive'
assert wrapper.num_classes == 1000,            f'Expected 1000 ImageNet classes, got {wrapper.num_classes}'
assert wrapper.config.patch_mode is True,      'patch_mode must be True to extract patch tokens'
assert wrapper.target_layers == TARGET_LAYERS, 'Target-layer mismatch between config and wrapper'
print(f'SC 2 ✓  d_model={wrapper.d_model}, num_classes={wrapper.num_classes}, patch_mode=True')

## Load ImageNet val dataset and sample 10 random images

In [ ]:
transform   = wrapper.get_transform()
dataset     = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=transform)
raw_dataset = datasets.ImageFolder(IMAGENET_VAL_DIR)  # no transform — PIL Images for viz
class_names = dataset.classes

rng            = random.Random(SEED)
sample_indices = rng.sample(range(len(dataset)), N_IMAGES)

print(f'Val images : {len(dataset):,}  ({len(class_names)} classes)')
print(f'Sampled idx: {sample_indices}')

## SC 3 — Verify dataset loading and display sampled images

In [ ]:
img_chk, lbl_chk = dataset[sample_indices[0]]
assert img_chk.shape == (3, 224, 224), f'Expected (3,224,224), got {img_chk.shape}'
assert 0 <= lbl_chk < len(class_names), f'Label {lbl_chk} out of range [0, {len(class_names)})'
print(f'Tensor shape: {tuple(img_chk.shape)}   label: {lbl_chk} ({class_names[lbl_chk]})')

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for ax, idx in zip(axes.flat, sample_indices):
    pil_img, lbl = raw_dataset[idx]
    ax.imshow(pil_img.resize((224, 224)))
    ax.set_title(f'{class_names[lbl][:18]}\nidx={idx}', fontsize=8)
    ax.axis('off')
plt.suptitle('10 Sampled ImageNet Val Images', fontsize=13)
plt.tight_layout()
plt.show()
print('SC 3 ✓')

## Extract patch tokens for all sampled images

In [ ]:
all_patch_states = {}  # {img_idx: {layer_idx: Tensor[1, H, W, d_model]}}
all_logits       = {}  # {img_idx: Tensor[1, num_classes]}
all_y_hats       = {}  # {img_idx: int}

for n, idx in enumerate(sample_indices):
    img_t, _ = dataset[idx]
    batch     = img_t.unsqueeze(0).to(DEVICE)         # [1, 3, 224, 224]

    patch_states, logits = wrapper.extract_patches(batch)

    all_patch_states[idx] = {k: v.cpu() for k, v in patch_states.items()}
    all_logits[idx]       = logits.cpu()
    all_y_hats[idx]       = int(logits[0].argmax())

    pred = class_names[all_y_hats[idx]]
    print(f'  [{n+1:2d}/{N_IMAGES}] idx={idx:6d}  y_hat={all_y_hats[idx]:4d}  ({pred})')

print('Extraction complete.')

## SC 4 — Verify patch tensor shapes and value ranges

In [ ]:
idx0 = sample_indices[0]
print(f'Checking shapes for image idx={idx0}:')
for layer_idx in TARGET_LAYERS:
    ps  = all_patch_states[idx0][layer_idx]
    exp = (1, H, W, wrapper.d_model)
    assert ps.shape == exp, f'Layer {layer_idx}: expected {exp}, got {ps.shape}'
    print(f'  Layer {layer_idx:2d}: shape={tuple(ps.shape)}  range=[{ps.min():.3f}, {ps.max():.3f}]  ✓')

lg = all_logits[idx0]
assert lg.shape == (1, wrapper.num_classes), f'Logits shape: {lg.shape}'
print(f'\nLogits shape  : {tuple(lg.shape)}')
print(f'y_hat for idx0: {all_y_hats[idx0]}  ({class_names[all_y_hats[idx0]]})')
print('SC 4 ✓')

## Load patch lens checkpoints

In [ ]:
lenses = {}   # {layer_idx: BaseLens}
for layer_idx in TARGET_LAYERS:
    lenses[layer_idx] = load_lens_checkpoint(available_lenses[layer_idx], device=DEVICE)
    print(f'  Layer {layer_idx:2d}: {type(lenses[layer_idx]).__name__}')
print('Patch lenses loaded.')

## SC 5 — Verify lens input/output dimensions

In [ ]:
k            = PATCH_NEIGHBOR_SIZE
expected_in  = k * k * wrapper.d_model
expected_out = wrapper.num_classes
print(f'Expected lens input  = k*k*d_model = {k}*{k}*{wrapper.d_model} = {expected_in}')
print(f'Expected lens output = num_classes = {expected_out}\n')

for layer_idx, lens in lenses.items():
    if hasattr(lens, 'linear'):   # AffineLens
        w_in  = lens.linear.weight.shape[1]
        w_out = lens.linear.weight.shape[0]
    else:                          # MLPLens
        lins  = [m for m in lens.net.modules() if hasattr(m, 'weight') and m.weight.dim() == 2]
        w_in  = lins[0].weight.shape[1]
        w_out = lins[-1].weight.shape[0]

    in_ok  = 'OK' if w_in  == expected_in  else f'MISMATCH (got {w_in})'
    out_ok = 'OK' if w_out == expected_out else f'MISMATCH (got {w_out})'
    print(f'  Layer {layer_idx:2d}: in={w_in} [{in_ok}]   out={w_out} [{out_ok}]')
    assert w_in  == expected_in,  f'Layer {layer_idx} input dim mismatch'
    assert w_out == expected_out, f'Layer {layer_idx} output dim mismatch'

print('\nSC 5 ✓')

## Compute k×k neighborhood score maps

For each valid center patch `(i, j)` (not within `border` of the edge):
- Extract the `k×k` neighborhood via zero-padded grid
- Flatten → feed to patch lens → softmax → `score[i,j] = prob[y_hat]`

In [ ]:
k      = PATCH_NEIGHBOR_SIZE
half_k = k // 2
border = PATCH_BORDER

all_score_maps = {}  # {img_idx: {layer_idx: Tensor[H, W]}}

for img_idx in sample_indices:
    all_score_maps[img_idx] = {}
    y_hat = all_y_hats[img_idx]

    for layer_idx in TARGET_LAYERS:
        patches = all_patch_states[img_idx][layer_idx].to(DEVICE)   # [1, H, W, d]
        _, H_p, W_p, d = patches.shape

        # Zero-pad spatially so every center in [border, H-border) has a full k×k neighborhood
        padded = F.pad(
            patches.permute(0, 3, 1, 2),            # [1, d, H, W]
            (half_k, half_k, half_k, half_k),
        ).permute(0, 2, 3, 1)                        # [1, H+2hk, W+2hk, d]

        score_map = torch.full((H_p, W_p), float('nan'))
        positions = []
        nbs       = []

        for i in range(border, H_p - border):
            for j in range(border, W_p - border):
                pi, pj = i + half_k, j + half_k
                nb = padded[0, pi-half_k : pi+half_k+1, pj-half_k : pj+half_k+1, :]
                nbs.append(nb.reshape(k * k * d))
                positions.append((i, j))

        if nbs:
            nb_batch = torch.stack(nbs)                        # [N_valid, k*k*d]
            with torch.no_grad():
                out_logits = lenses[layer_idx](nb_batch)       # [N_valid, C]
            scores = F.softmax(out_logits, dim=-1)[:, y_hat].cpu()
            for (i, j), s in zip(positions, scores.tolist()):
                score_map[i, j] = s

        all_score_maps[img_idx][layer_idx] = score_map

print('Scoring complete for all images and layers.')

## SC 6 — Verify score map shapes, NaN counts, and value ranges

In [ ]:
n_valid_exp = (H - 2 * PATCH_BORDER) * (W - 2 * PATCH_BORDER)
n_nan_exp   = H * W - n_valid_exp
print(f'Grid {H}x{W}: expected {n_valid_exp} valid patches, {n_nan_exp} border (NaN)\n')

idx0 = sample_indices[0]
for layer_idx in TARGET_LAYERS:
    sm    = all_score_maps[idx0][layer_idx]
    assert sm.shape == (H, W), f'Layer {layer_idx}: score map shape {sm.shape}'

    valid = sm[~torch.isnan(sm)]
    n_nan = int(torch.isnan(sm).sum())

    assert valid.numel() == n_valid_exp, (
        f'Layer {layer_idx}: {valid.numel()} valid patches, expected {n_valid_exp}'
    )
    assert n_nan == n_nan_exp, (
        f'Layer {layer_idx}: {n_nan} NaN patches, expected {n_nan_exp}'
    )
    assert (valid >= 0.0).all() and (valid <= 1.0).all(), (
        f'Layer {layer_idx}: scores out of [0,1] — [{valid.min():.4f}, {valid.max():.4f}]'
    )
    print(
        f'  Layer {layer_idx:2d}  valid={valid.numel():4d}/{H*W}  '
        f'score=[{valid.min():.4f},{valid.max():.4f}]  mean={valid.mean():.4f}  ✓'
    )

print('\nSC 6 ✓')

## Visualize: score grid (patch resolution) + heatmap overlay on original image

Border patches (NaN) are filled with `nanmin` for display purposes.

In [ ]:
k = PATCH_NEIGHBOR_SIZE

for img_idx in sample_indices:
    y_hat       = all_y_hats[img_idx]
    pil_img, _  = raw_dataset[img_idx]
    img_np      = np.array(pil_img.resize((224, 224)))

    n_cols = len(TARGET_LAYERS) + 1
    fig, axes = plt.subplots(2, n_cols, figsize=(3.6 * n_cols, 8))

    # ── Column 0: original image ───────────────────────────────────────────
    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title(f'Original  idx={img_idx}', fontsize=8)
    axes[0, 0].axis('off')
    axes[1, 0].text(
        0.5, 0.5, f'Pred:\n{class_names[y_hat][:22]}',
        ha='center', va='center', fontsize=8,
        transform=axes[1, 0].transAxes
    )
    axes[1, 0].axis('off')

    # ── Columns 1..N: per-layer score grid + overlay ───────────────────────
    for col, layer_idx in enumerate(TARGET_LAYERS, start=1):
        sm       = all_score_maps[img_idx][layer_idx].numpy()
        nan_fill = float(np.nanmin(sm)) if not np.all(np.isnan(sm)) else 0.0
        sm_plot  = np.where(np.isnan(sm), nan_fill, sm)
        vmax_val = float(np.nanmax(sm)) if not np.all(np.isnan(sm)) else 1.0

        # Top row: score grid at patch resolution
        im = axes[0, col].imshow(sm_plot, cmap='hot', vmin=0.0, vmax=vmax_val)
        axes[0, col].set_title(f'Layer {layer_idx}  Score Grid ({H}x{W})', fontsize=8)
        axes[0, col].axis('off')
        plt.colorbar(im, ax=axes[0, col], fraction=0.046, pad=0.04)

        # Bottom row: normalized heatmap overlaid on 224×224 image
        hm_norm = (sm_plot - sm_plot.min()) / (sm_plot.max() - sm_plot.min() + 1e-8)
        hm_u8   = (hm_norm * 255).astype(np.uint8)
        hm_224  = np.array(Image.fromarray(hm_u8).resize((224, 224), Image.NEAREST))
        axes[1, col].imshow(img_np)
        axes[1, col].imshow(hm_224, cmap='hot', alpha=0.55, vmin=0, vmax=255)
        axes[1, col].set_title(f'Layer {layer_idx}  Overlay', fontsize=8)
        axes[1, col].axis('off')

    title = (
        f'Patch Lens ({k}x{k} neighborhoods)  '
        f'idx={img_idx}  Pred: {class_names[y_hat]}'
    )
    plt.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()